In [1]:
import pandas as pd
import numpy as np

In [3]:
options_data = pd.read_hdf("C:\\Users\\asus\Downloads\\all_options_data.h5", key="AAPL").set_index("date")
options_data['exdate'] = pd.to_datetime(options_data['exdate'])
options_data.index = pd.to_datetime(options_data.index)

price_data = pd.read_hdf("C:\\Users\\asus\Downloads\\all_tickers_time_series.hf5", key="AAPL").drop_duplicates().set_index("date")
price_series = price_data["prc"] # subject to changes

treasury_bill_3m = pd.read_csv("C:\\Users\\asus\Downloads\\DTB3.csv").set_index("DATE")
treasury_bill_3m.index = pd.to_datetime(treasury_bill_3m.index)

options_data_filtered = options_data[options_data.index.isin(price_series.index)]
options_data_filtered['price'] = price_series
options_data_filtered["strike_price"] = options_data_filtered["strike_price"] / 1000
options_data_filtered["risk-free interest rate %"] = treasury_bill_3m["DTB3"]
options_data_filtered["risk-free interest rate %"] = pd.to_numeric(options_data_filtered["risk-free interest rate %"], errors='coerce')
options_data_filtered["days to expiry"] = (options_data_filtered["exdate"] - options_data_filtered.index).dt.days
options_data_filtered = options_data_filtered[options_data_filtered["days to expiry"] != 0] # remove options expiring that day
options_data_filtered["impl_volatility"].fillna(method='ffill', inplace=True) # forward fill when sigma=0
options_data_filtered["risk-free interest rate %"].fillna(method='ffill', inplace=True) # forward fill when t-bill yield unknown
options_data_filtered["strike minus price"] = np.abs(options_data_filtered["strike_price"] - options_data_filtered["price"])

options_data_filtered = options_data_filtered[
    options_data_filtered.apply(
        lambda row: row["strike_price"] < row["price"] if row["cp_flag"] == 'C' else row["strike_price"] > row["price"],
        axis=1
    )
]

options_data_filtered

date_index = options_data_filtered.index.unique()
filtered_data = pd.DataFrame()
for date in date_index:
    options_on_date = options_data_filtered.loc[date]
    no_of_ex_dates = options_on_date["exdate"].nunique()
    if no_of_ex_dates > 1:
        options_on_date = options_on_date.query("exdate == exdate.min()")
    closest_strike = options_on_date[options_on_date["strike minus price"] == options_on_date["strike minus price"].min()]
    if len(closest_strike) > 2:
        c_closest_strike = closest_strike[closest_strike["cp_flag"] == "C"]
        c_closest_strike = c_closest_strike[c_closest_strike["open_interest"] == c_closest_strike["open_interest"].max()]
        p_closest_strike = closest_strike[closest_strike["cp_flag"] == "P"]
        p_closest_strike = p_closest_strike[p_closest_strike["open_interest"] == p_closest_strike["open_interest"].max()]        
        closest_strike = pd.concat([c_closest_strike, p_closest_strike])        
    filtered_data = pd.concat([filtered_data, closest_strike])

C:\Users\asus\AppData\Local\Temp\ipykernel_6600\3041055593.py:18: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  options_data_filtered["impl_volatility"].fillna(method='ffill', inplace=True) # forward fill when sigma=0
C:\Users\asus\AppData\Local\Temp\ipykernel_6600\3041055593.py:18: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  options_data_filtered["impl_volatility"].fillna(method='ffill', inplace=True) # forward fill when sigma=0
C:\User

In [4]:
options_data_filtered.head()

,ticker,exdate,cp_flag,strike_price,best_bid,best_offer,open_interest,impl_volatility,delta,gamma,theta,vega,volume,price,risk-free interest rate %,days to expiry,strike minus price
date,,,,,,,,,,,,,,,,,
2000-01-10,AAPL,2000-01-22,P,100.0,7.250,7.500,4063.0,0.879421,-0.521674,0.025665,-91.488500,7.057306,48.0,97.75,5.24,12,2.25
2000-01-10,AAPL,2000-01-22,C,40.0,57.625,58.125,1104.0,2.002907,0.995955,0.000338,-8.733804,0.212873,7.0,97.75,5.24,12,57.75
2000-01-10,AAPL,2000-01-22,C,90.0,10.750,11.250,2273.0,0.924280,0.721622,0.020485,-87.013050,5.950613,69.0,97.75,5.24,12,7.75
2000-01-10,AAPL,2000-01-22,P,105.0,10.500,10.875,792.0,0.894597,-0.637983,0.023814,-86.876010,6.640641,5.0,97.75,5.24,12,7.25
2000-01-10,AAPL,2000-01-22,C,80.0,18.750,19.000,5429.0,0.973746,0.891338,0.010802,-52.834830,3.299727,117.0,97.75,5.24,12,17.75


In [5]:
from scipy.stats import norm
from math import log, sqrt, exp

def Black76LognormalCall(S, K, r, sigma, T):
    if T <= 0 or S <= 0 or K <= 0 or sigma < 0:
        return float('nan')
    
    d1 = (log(S/K)+(r+sigma**2/2)*T) / (sigma*sqrt(T))
    d2 = d1 - sigma*sqrt(T)
    
    call_price = S*norm.cdf(d1) - K*exp(-r*T)*norm.cdf(d2)
    return call_price

def Black76LognormalPut(S, K, r, sigma, T):
    if T <= 0 or S <= 0 or K <= 0 or sigma < 0:
        return float('nan')
    
    d1 = (log(S/K)+(r+sigma**2/2)*T) / (sigma*sqrt(T))
    d2 = d1 - sigma*sqrt(T)
    
    put_price = K*exp(-r*T)*norm.cdf(-d2) - S*norm.cdf(-d1)
    return put_price

In [6]:
options_data_filtered['option_price'] = options_data_filtered.apply(
    lambda row: Black76LognormalCall(row['price'], 
                                      row['strike_price'], 
                                      row['risk-free interest rate %'] / 100, 
                                      row['impl_volatility'], 
                                      row['days to expiry'] / 365) if row['cp_flag'] == 'C' 
                else Black76LognormalPut(row['price'], 
                                         row['strike_price'], 
                                         row['risk-free interest rate %'] / 100, 
                                         row['impl_volatility'], 
                                         row['days to expiry'] / 365),
    axis=1
)

options_data_filtered


,ticker,exdate,cp_flag,strike_price,best_bid,best_offer,open_interest,impl_volatility,delta,gamma,theta,vega,volume,price,risk-free interest rate %,days to expiry,strike minus price,option_price
date,,,,,,,,,,,,,,,,,,
2000-01-10,AAPL,2000-01-22,P,100.0,7.250,7.500,4063.0,0.879421,-0.521674,0.025665,-91.488500,7.057306,48.0,97.75,5.24,12,2.25,7.371043
2000-01-10,AAPL,2000-01-22,C,40.0,57.625,58.125,1104.0,2.002907,0.995955,0.000338,-8.733804,0.212873,7.0,97.75,5.24,12,57.75,57.868795
2000-01-10,AAPL,2000-01-22,C,90.0,10.750,11.250,2273.0,0.924280,0.721622,0.020485,-87.013050,5.950613,69.0,97.75,5.24,12,7.75,10.990544
2000-01-10,AAPL,2000-01-22,P,105.0,10.500,10.875,792.0,0.894597,-0.637983,0.023814,-86.876010,6.640641,5.0,97.75,5.24,12,7.25,10.677517
2000-01-10,AAPL,2000-01-22,C,80.0,18.750,19.000,5429.0,0.973746,0.891338,0.010802,-52.834830,3.299727,117.0,97.75,5.24,12,17.75,18.864337
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2023-08-31,AAPL,2023-09-01,P,265.0,76.900,77.000,0.0,0.742784,NaN,NaN,NaN,NaN,2.0,187.87,5.32,1,77.13,77.091378
2023-08-31,AAPL,2023-09-01,P,245.0,55.950,57.550,0.0,4.687440,NaN,NaN,NaN,NaN,0.0,187.87,5.32,1,57.13,60.821384
2023-08-31,AAPL,2023-09-01,C,90.0,97.800,98.600,0.0,6.783724,0.987817,0.000475,-390.293100,0.312062,0.0,187.87,5.32,1,97.87,98.200738


In [7]:
from scipy.optimize import brentq

def impliedCallVolatility(S, K, r, price, T):
    # Evaluate at endpoints
    f_a = price - Black76LognormalCall(S, K, r, 1e-6, T)
    f_b = price - Black76LognormalCall(S, K, r, 5, T)

    if f_a * f_b > 0:
        return float('nan')  # Or handle as appropriate

    # Proceed with brentq if valid
    impliedVol = brentq(lambda x: price - Black76LognormalCall(S, K, r, x, T), 1e-6, 5)
    
    return impliedVol

def impliedPutVolatility(S, K, r, price, T):
    # Evaluate at endpoints
    f_a = price - Black76LognormalPut(S, K, r, 1e-6, T)
    f_b = price - Black76LognormalPut(S, K, r, 5, T)

    if f_a * f_b > 0:
        return float('nan')  # Or handle as appropriate

    # Proceed with brentq if valid
    impliedVol = brentq(lambda x: price - Black76LognormalPut(S, K, r, x, T), 1e-6, 5)
    
    return impliedVol

In [8]:
options_data_filtered['implied volatility'] = options_data_filtered.apply(
    lambda row: impliedCallVolatility(row['price'], 
                                     row['strike_price'], 
                                     row['risk-free interest rate %'] / 100, 
                                     row['option_price'], 
                                     row['days to expiry'] / 365) if row['cp_flag'] == 'C'
                else impliedPutVolatility(row['price'], 
                                     row['strike_price'], 
                                     row['risk-free interest rate %'] / 100, 
                                     row['option_price'], 
                                     row['days to expiry'] / 365), axis=1)

options_data_filtered

,ticker,exdate,cp_flag,strike_price,best_bid,best_offer,open_interest,impl_volatility,delta,gamma,theta,vega,volume,price,risk-free interest rate %,days to expiry,strike minus price,option_price,implied volatility
date,,,,,,,,,,,,,,,,,,,
2000-01-10,AAPL,2000-01-22,P,100.0,7.250,7.500,4063.0,0.879421,-0.521674,0.025665,-91.488500,7.057306,48.0,97.75,5.24,12,2.25,7.371043,0.879421
2000-01-10,AAPL,2000-01-22,C,40.0,57.625,58.125,1104.0,2.002907,0.995955,0.000338,-8.733804,0.212873,7.0,97.75,5.24,12,57.75,57.868795,2.002907
2000-01-10,AAPL,2000-01-22,C,90.0,10.750,11.250,2273.0,0.924280,0.721622,0.020485,-87.013050,5.950613,69.0,97.75,5.24,12,7.75,10.990544,0.924280
2000-01-10,AAPL,2000-01-22,P,105.0,10.500,10.875,792.0,0.894597,-0.637983,0.023814,-86.876010,6.640641,5.0,97.75,5.24,12,7.25,10.677517,0.894597
2000-01-10,AAPL,2000-01-22,C,80.0,18.750,19.000,5429.0,0.973746,0.891338,0.010802,-52.834830,3.299727,117.0,97.75,5.24,12,17.75,18.864337,0.973746
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2023-08-31,AAPL,2023-09-01,P,265.0,76.900,77.000,0.0,0.742784,NaN,NaN,NaN,NaN,2.0,187.87,5.32,1,77.13,77.091378,0.000001
2023-08-31,AAPL,2023-09-01,P,245.0,55.950,57.550,0.0,4.687440,NaN,NaN,NaN,NaN,0.0,187.87,5.32,1,57.13,60.821384,4.687440
2023-08-31,AAPL,2023-09-01,C,90.0,97.800,98.600,0.0,6.783724,0.987817,0.000475,-390.293100,0.312062,0.0,187.87,5.32,1,97.87,98.200738,NaN
